# NemoNexus Local Evaluation - Quick Test

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
import os
from typing import Dict, List, Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
else:
    print('No GPU available - using CPU')

In [ ]:
class SpecialistModel(nn.Module):
    def __init__(self, max_grid_size=30, d_model=256, specialist_type='MINERVA'):
        super().__init__()
        self.specialist_type = specialist_type
        self.max_grid_size = max_grid_size
        self.d_model = d_model
        
        if specialist_type == 'MINERVA':
            self.num_layers = 6
            self.num_heads = 8
            self.d_model = 1024
        elif specialist_type == 'ATLAS':
            self.num_layers = 4
            self.num_heads = 8
            self.d_model = 512
        elif specialist_type == 'IRIS':
            self.num_layers = 3
            self.num_heads = 4
            self.d_model = 512
        elif specialist_type == 'CHRONOS':
            self.num_layers = 8
            self.num_heads = 8
            self.d_model = 1024
        elif specialist_type == 'PROMETHEUS':
            self.num_layers = 6
            self.num_heads = 6
            self.d_model = 768
        
        self.input_embedding = nn.Embedding(10, self.d_model)
        self.position_embedding = nn.Parameter(torch.randn(max_grid_size * max_grid_size, self.d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=self.num_heads,
            dim_feedforward=self.d_model * 4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.num_layers)
        self.output_projection = nn.Linear(self.d_model, 10)
        
    def forward(self, x):
        B, H, W = x.shape
        x_flat = x.view(B, -1)
        embedded = self.input_embedding(x_flat)
        seq_len = embedded.shape[1]
        embedded += self.position_embedding[:seq_len].unsqueeze(0)
        output = self.transformer(embedded)
        logits = self.output_projection(output)
        return logits.view(B, H, W, 10)

In [ ]:
class TaskRouter(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )
        
    def forward(self, input_features):
        weights = self.feature_extractor(input_features)
        return F.softmax(weights, dim=-1)

In [ ]:
class OLYMPUSEnsemble(nn.Module):
    def __init__(self, max_grid_size=30, d_model=256):
        super().__init__()
        self.max_grid_size = max_grid_size
        
        self.specialists = nn.ModuleDict({
            'MINERVA': SpecialistModel(max_grid_size, d_model, 'MINERVA'),
            'ATLAS': SpecialistModel(max_grid_size, d_model, 'ATLAS'),
            'IRIS': SpecialistModel(max_grid_size, d_model, 'IRIS'),
            'CHRONOS': SpecialistModel(max_grid_size, d_model, 'CHRONOS'),
            'PROMETHEUS': SpecialistModel(max_grid_size, d_model, 'PROMETHEUS')
        })
        
        self.task_router = TaskRouter(d_model)
        
    def forward(self, input_grid, train_examples=None, target_shape=None):
        B, H, W = input_grid.shape
        
        specialist_outputs = {}
        for name, specialist in self.specialists.items():
            with torch.no_grad():
                output = specialist(input_grid)
                specialist_outputs[name] = output
        
        ensemble_output = torch.stack(list(specialist_outputs.values())).mean(dim=0)
        return ensemble_output
    
    def load_trained_weights(self, model_path):
        try:
            if os.path.exists(model_path):
                checkpoint = torch.load(model_path, map_location=device)
                # Load from ensemble_state_dict key (olympus_v3_best.pt format)
                if 'ensemble_state_dict' in checkpoint:
                    self.load_state_dict(checkpoint['ensemble_state_dict'], strict=False)
                    print(f'Loaded NemoNexus ensemble_state_dict from {model_path}')
                    return True
                else:
                    self.load_state_dict(checkpoint, strict=False)
                    print(f'Loaded NemoNexus direct state_dict from {model_path}')
                    return True
        except Exception as e:
            print(f'Could not load NemoNexus weights: {e}')
        
        print('Using randomly initialized weights')
        return False

In [ ]:
class NemoNexus:
    def __init__(self, model_path=None):
        self.olympus = OLYMPUSEnsemble().to(device)
        
        if model_path:
            self.olympus.load_trained_weights(model_path)
        
    def evaluate_task(self, task_data):
        """Evaluate a single task and return accuracy (only works if outputs available)"""
        train_examples = task_data['train']
        test_cases = task_data['test']
        
        self.olympus.eval()
        correct = 0
        total = len(test_cases)
        
        for test_case in test_cases:
            input_grid = np.array(test_case['input'])
            
            # Check if output is available (training data has it, evaluation data doesn't)
            if 'output' not in test_case:
                print("Warning: No 'output' key found - this is evaluation data without ground truth")
                return None
                
            target_grid = np.array(test_case['output'])
            
            input_tensor = torch.tensor(input_grid, dtype=torch.long).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output_logits = self.olympus(input_tensor, train_examples)
                output_pred = torch.argmax(output_logits, dim=-1)
                pred_grid = output_pred[0].cpu().numpy()
            
            # Check exact match
            if pred_grid.shape == target_grid.shape and np.array_equal(pred_grid, target_grid):
                correct += 1
        
        return correct / total if total > 0 else 0.0
    
    def predict_task(self, task_data):
        """Generate predictions for a task (works for evaluation data)"""
        train_examples = task_data['train']
        test_cases = task_data['test']
        
        self.olympus.eval()
        predictions = []
        
        for test_case in test_cases:
            input_grid = np.array(test_case['input'])
            input_tensor = torch.tensor(input_grid, dtype=torch.long).unsqueeze(0).to(device)
            
            with torch.no_grad():
                output_logits = self.olympus(input_tensor, train_examples)
                output_pred = torch.argmax(output_logits, dim=-1)
                pred_grid = output_pred[0].cpu().numpy()
            
            predictions.append(pred_grid.tolist())
        
        return predictions
    
    def evaluate_dataset(self, dataset_path, max_tasks=None):
        """Evaluate on a full dataset (only works if ground truth available)"""
        with open(dataset_path, 'r') as f:
            data = json.load(f)
        
        task_ids = list(data.keys())
        if max_tasks:
            task_ids = task_ids[:max_tasks]
        
        total_accuracy = 0.0
        task_results = {}
        valid_tasks = 0
        
        print(f'Evaluating {len(task_ids)} tasks...')
        
        for i, task_id in enumerate(task_ids):
            task_data = data[task_id]
            accuracy = self.evaluate_task(task_data)
            
            if accuracy is not None:  # Only count tasks with ground truth
                task_results[task_id] = accuracy
                total_accuracy += accuracy
                valid_tasks += 1
            else:
                print(f"Skipping {task_id} - no ground truth available")
                break
            
            if (i + 1) % 10 == 0:
                print(f'Processed {i + 1}/{len(task_ids)} tasks, Current avg: {total_accuracy / valid_tasks:.3f}')
        
        if valid_tasks == 0:
            print("No tasks with ground truth found - this appears to be evaluation data")
            return None, {}
        
        overall_accuracy = total_accuracy / valid_tasks
        return overall_accuracy, task_results

In [ ]:
# Load NemoNexus from your local path
nemo = NemoNexus(model_path='/content/drive/MyDrive/NemoNexus/NemoNexus.pt')
print('NemoNexus loaded and ready for evaluation!')

In [ ]:
# Evaluate on evaluation dataset (if available)
eval_path = '/content/drive/MyDrive/AutomataNexus_Olympus_AGI2/data/arc-agi_evaluation_challenges.json'
if os.path.exists(eval_path):
    print('\n=== Evaluating on ARC Evaluation Set ===')
    eval_accuracy, eval_results = nemo.evaluate_dataset(eval_path, max_tasks=50)  # Limit for speed
    print(f'\nEvaluation Accuracy: {eval_accuracy:.3f} ({eval_accuracy*100:.1f}%)')
    
    # Show some individual results
    print('\nSample task results:')
    for i, (task_id, acc) in enumerate(list(eval_results.items())[:5]):
        print(f'  {task_id}: {acc:.3f}')
else:
    print('Evaluation dataset not found at:', eval_path)

In [ ]:
# Evaluate on training dataset (for comparison)
train_path = '/content/drive/MyDrive/AutomataNexus_Olympus_AGI2/data/arc-agi_training_challenges.json'
if os.path.exists(train_path):
    print('\n=== Evaluating on ARC Training Set (Sample) ===')
    train_accuracy, train_results = nemo.evaluate_dataset(train_path, max_tasks=25)  # Small sample
    print(f'\nTraining Sample Accuracy: {train_accuracy:.3f} ({train_accuracy*100:.1f}%)')
    
    # Show some individual results
    print('\nSample task results:')
    for i, (task_id, acc) in enumerate(list(train_results.items())[:5]):
        print(f'  {task_id}: {acc:.3f}')
else:
    print('Training dataset not found at:', train_path)

In [ ]:
# Quick single task test
print('\n=== Testing Single Task ===')
# Load any available dataset for a quick test
test_files = ['/content/drive/MyDrive/AutomataNexus_Olympus_AGI2/data/arc-agi_training_challenges.json',
              '/content/drive/MyDrive/AutomataNexus_Olympus_AGI2/data/arc-agi_evaluation_challenges.json']

for test_file in test_files:
    if os.path.exists(test_file):
        with open(test_file, 'r') as f:
            data = json.load(f)
        
        # Test first task
        first_task_id = list(data.keys())[0]
        first_task = data[first_task_id]
        
        print(f'Testing task: {first_task_id}')
        print(f'Training examples: {len(first_task["train"])}')
        print(f'Test cases: {len(first_task["test"])}')
        
        accuracy = nemo.evaluate_task(first_task)
        print(f'Task accuracy: {accuracy:.3f}')
        break
else:
    print('No datasets found for testing')

In [ ]:
# Quick single task test
print('\n=== Testing Single Task ===')
# Load any available dataset for a quick test
test_files = ['/content/AutomataNexus_Olympus_AGI2/data/arc-agi_training_challenges.json',
              '/content/AutomataNexus_Olympus_AGI2/data/arc-agi_evaluation_challenges.json']

for test_file in test_files:
    if os.path.exists(test_file):
        with open(test_file, 'r') as f:
            data = json.load(f)
        
        # Test first task
        first_task_id = list(data.keys())[0]
        first_task = data[first_task_id]
        
        print(f'Testing task: {first_task_id}')
        print(f'Training examples: {len(first_task["train"])}')
        print(f'Test cases: {len(first_task["test"])}')
        
        accuracy = nemo.evaluate_task(first_task)
        print(f'Task accuracy: {accuracy:.3f}')
        break
else:
    print('No datasets found for testing')